# Feature Engineering and Data Splitting

This notebook handles the feature engineering and preprocessing for the MMP-9 bioactivity dataset. The key steps include:
1. Loading the processed bioactivity data.
2. Grouping compounds by Murcko Scaffolds to perform deterministic, structurally-aware train/test splitting.
3. Calculating Morgan fingerprints (ECFP4) for molecular feature representation.
4. Handling class imbalances in the training set using SMOTE.

In [1]:
# Import data manipulation and numerical libraries
import pandas as pd
import numpy as np

# Import RDKit for cheminformatics operations
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Scaffolds import MurckoScaffold

# Additional utilities
from collections import defaultdict
import warnings

# Suppress deprecation warnings from RDKit for cleaner output logs
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Load Processed Data
Here we load the cleaned dataset containing the canonical SMILES strings and bioactivity classifications. We also verify the dataset's shape and the overall distribution of active vs. inactive compounds.

In [2]:
# Load the processed bioactivity data
df = pd.read_csv('../data/processed/mmp9_processed.csv')

# Print the shape of the dataset to verify successful loading
print(f"Dataset loaded successfully with shape: {df.shape}")

# Check the distribution of the target variable (bioactivity_class)
print("\nOverall Bioactivity Class Distribution:")
print(df['bioactivity_class'].value_counts())

Dataset loaded successfully with shape: (1661, 4)

Overall Bioactivity Class Distribution:
bioactivity_class
1    1227
0     434
Name: count, dtype: int64


## 2. Murcko Scaffold Splitting
To evaluate our model's ability to generalize to novel, unseen chemical spaces, we use **Scaffold Splitting** rather than a random split. 
- Compounds sharing the exact same Murcko Scaffold are grouped together.
- Scaffolds are sorted by size and sequentially distributed into the training and testing sets to achieve an approximate 80/20 split.

In [3]:
def get_murcko_scaffold(smiles):
    """
    Generate the Murcko scaffold for a given SMILES string.
    Returns the SMILES string of the scaffold frame.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Calculate the Murcko scaffold without chirality parameters
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    return scaffold

print("Calculating Murcko scaffolds for all compounds...")
# Assign a scaffold to each compound in the dataset
df['scaffold'] = df['canonical_smiles'].apply(get_murcko_scaffold)

# Group compound indices by their scaffold
scaffold_to_indices = defaultdict(list)
for idx, scaffold in enumerate(df['scaffold']):
    scaffold_to_indices[scaffold].append(idx)

# Sort scaffolds by the number of compounds they contain (largest first) to ensure deterministic splitting
scaffold_sets = sorted(scaffold_to_indices.values(), key=len, reverse=True)

# Define the target split ratio
train_size = 0.8
train_indices, test_indices = [], []

print("Splitting data into 80/20 train/test sets based on scaffolds...")
# Distribute scaffold sets to train or test to maintain the target ratio
for scaffold_indices in scaffold_sets:
    if len(train_indices) / len(df) < train_size:
        train_indices.extend(scaffold_indices)
    else:
        test_indices.extend(scaffold_indices)

# Create the final training and testing dataframes
df_train = df.iloc[train_indices].copy()
df_test = df.iloc[test_indices].copy()

# Output the metrics of the split
print(f"Train set size: {len(df_train)} | Test set size: {len(df_test)}")
print(f"Train active %: {df_train['bioactivity_class'].mean()*100:.1f}%")
print(f"Test active %: {df_test['bioactivity_class'].mean()*100:.1f}%")

Calculating Murcko scaffolds for all compounds...
Splitting data into 80/20 train/test sets based on scaffolds...
Train set size: 1329 | Test set size: 332
Train active %: 75.8%
Test active %: 66.3%


## 3. Validation & Morgan Fingerprint Generation
First, we verify that the scaffold split was successful. There should be exactly zero structural overlap between the scaffolds in the training set and those in the testing set.

Next, we convert our SMILES strings into **Morgan Fingerprints** (Radius=2, 2048 bits). These fingerprints capture the circular topology of the molecules and serve as our numerical feature vectors (`X`) for machine learning.

In [4]:
# --- 1. Scaffold Overlap Check ---
# Extract the unique scaffolds from both splits
train_scaffolds = set(df_train['scaffold'])
test_scaffolds = set(df_test['scaffold'])

# Calculate the intersection of the two sets
overlap = train_scaffolds & test_scaffolds

# Print the number of overlapping scaffolds (expected to be 0)
print(f"Scaffold overlap between train and test: {len(overlap)}")
if len(overlap) == 0:
    print("Success: Train and Test sets have mutually exclusive scaffolds.\n")
else:
    print("Warning: There is data leakage between Train and Test sets.\n")

# --- 2. Morgan Fingerprint Generation ---
def generate_morgan_fingerprint(smiles, radius=2, n_bits=2048):
    """
    Generate a Morgan fingerprint for a SMILES string.
    Radius 2 is roughly equivalent to ECFP4.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,))
    # Generate the Morgan fingerprint bit vector
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

print("Generating Morgan fingerprints (Radius=2, 2048 bits) for training data...")
# Apply fingerprint generation to train set
X_train = np.stack(df_train['canonical_smiles'].apply(generate_morgan_fingerprint))
y_train = df_train['bioactivity_class'].values

print("Generating Morgan fingerprints for testing data...")
# Apply fingerprint generation to test set
X_test = np.stack(df_test['canonical_smiles'].apply(generate_morgan_fingerprint))
y_test = df_test['bioactivity_class'].values

print(f"Feature extraction complete. X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

Scaffold overlap between train and test: 0
Success: Train and Test sets have mutually exclusive scaffolds.

Generating Morgan fingerprints (Radius=2, 2048 bits) for training data...


[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerator
[15:22:10] DEPRECATION WARNING: please use MorganGenerat

Generating Morgan fingerprints for testing data...


[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerator
[15:22:15] DEPRECATION WARNING: please use MorganGenerat

Feature extraction complete. X_train shape: (1329, 2048), X_test shape: (332, 2048)


[15:22:16] DEPRECATION WARNING: please use MorganGenerator
[15:22:16] DEPRECATION WARNING: please use MorganGenerator
[15:22:16] DEPRECATION WARNING: please use MorganGenerator
[15:22:16] DEPRECATION WARNING: please use MorganGenerator
[15:22:16] DEPRECATION WARNING: please use MorganGenerator


## 4. Handling Class Imbalance with SMOTE
Biological datasets often have imbalanced classes (e.g., highly skewed towards active or inactive compounds). To prevent the predictive model from becoming biased towards the majority class, we apply the **Synthetic Minority Over-sampling Technique (SMOTE)** to synthetically balance the training features before modeling. 

*Note: SMOTE is strictly applied to the training set to prevent data leakage into the test set.*

In [5]:
from imblearn.over_sampling import SMOTE

print("Evaluating initial class balance...")
# Check distribution before resampling
class_counts_before = dict(zip(*np.unique(y_train, return_counts=True)))
print(f"Before SMOTE — train class distribution: {class_counts_before}")

# Initialize SMOTE and apply it exclusively to the training features
print("Applying SMOTE to balance the training set...")
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Check distribution after resampling
class_counts_after = dict(zip(*np.unique(y_train_balanced, return_counts=True)))
print(f"After SMOTE  — train class distribution: {class_counts_after}")
print(f"Balanced training features shape: {X_train_balanced.shape}")
print("\nPipeline finished! Data is ready for machine learning.")

Evaluating initial class balance...
Before SMOTE — train class distribution: {0: 322, 1: 1007}
Applying SMOTE to balance the training set...
After SMOTE  — train class distribution: {0: 1007, 1: 1007}
Balanced training features shape: (2014, 2048)

Pipeline finished! Data is ready for machine learning.


## 5. Save Model-Ready Data
Now that our features are extracted and our training data is balanced, we need to save the final arrays to disk. 

Instead of saving back to a CSV (which is inefficient for 2048-bit fingerprint arrays), we will save the data as a **compressed NumPy Archive (`.npz`)** in the original processed data directory. This ensures fast loading times and preserves the exact structure and data types of our matrices for the modeling phase.

In [6]:
import os

# Define the directory where the original dataset was loaded from
output_dir = '../data/processed/'

# Ensure the directory exists (best practice, though it should already exist)
os.makedirs(output_dir, exist_ok=True)

# Define the target file path
output_file = os.path.join(output_dir, 'mmp9_model_ready_splits.npz')

print(f"Preparing to save dataset splits to: {output_file}")

# Save the feature matrices and target arrays into a single compressed .npz file
# We use the balanced training sets and the untouched testing sets
np.savez_compressed(
    output_file,
    X_train=X_train_balanced,
    y_train=y_train_balanced,
    X_test=X_test,
    y_test=y_test
)

# Verify the file was created and print its size
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Success! Dataset saved successfully.")
print(f"File size: {file_size_mb:.2f} MB")

Preparing to save dataset splits to: ../data/processed/mmp9_model_ready_splits.npz
Success! Dataset saved successfully.
File size: 0.19 MB
